# ChatBS Base Token Usage Analysis

This notebook inspects raw `RESULTS.jsonl` files to analyze LLM token usage across methods. It uses the raw result paths referenced by `analysis/*/results.csv`, so the comparison matches the evaluated runs.

In [ ]:
from __future__ import annotations

import json
import math
import re
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

NOTEBOOK_DIR = Path.cwd()
BASE_DIR = NOTEBOOK_DIR if (NOTEBOOK_DIR / "analysis").exists() else Path("evaluations/chatbs-base")
ANALYSIS_DIR = BASE_DIR / "analysis"
EXPLAINER_DIR = BASE_DIR / "explainer"

BASE_DIR.resolve()

## 1. Discover Raw Result Files

In [ ]:
TASKS = ["bool", "entity", "numeric"]

def normalize_method(raw_method: str) -> str:
    return "ours" if raw_method == "results" else raw_method


def resolve_path(path_value: str) -> Path:
    path = Path(path_value)
    if path.exists():
        return path
    return (Path.cwd() / path).resolve()


analysis_paths = []
for task in TASKS:
    df = pd.read_csv(ANALYSIS_DIR / task / "results.csv")
    for run, path in df[["run", "prediction_path"]].drop_duplicates().itertuples(index=False):
        analysis_paths.append({"task": task, "method": run, "path": str(resolve_path(path))})

selected_raw_files = (
    pd.DataFrame(analysis_paths)
    .drop_duplicates(["method", "path"])
    .sort_values(["method", "path"])
    .reset_index(drop=True)
)

selected_raw_files

In [ ]:
# Optional: inspect every raw result file found under explainer/, including older experiments.
all_raw_files = pd.DataFrame(
    {
        "method": normalize_method(path.parts[-3]),
        "experiment": path.parts[-2],
        "path": str(path.resolve()),
    }
    for path in sorted(EXPLAINER_DIR.glob("*/*/RESULTS.jsonl"))
)
all_raw_files

## 2. Parse Raw Token Usage

In [ ]:
TOKEN_KEYS = ("prompt_tokens", "completion_tokens", "total_tokens")

def number(value: Any) -> float:
    if value is None or value == "":
        return 0.0
    try:
        if isinstance(value, float) and math.isnan(value):
            return 0.0
        return float(value)
    except (TypeError, ValueError):
        return 0.0


def nested_get(mapping: Any, *keys: str) -> Any:
    cur = mapping
    for key in keys:
        if not isinstance(cur, dict):
            return None
        cur = cur.get(key)
    return cur


def extract_reasoning_tokens(usage: dict[str, Any]) -> float:
    candidates = [
        nested_get(usage, "completion_tokens_details", "reasoning_tokens"),
        nested_get(usage, "completion_tokens_details", "reasoning"),
        nested_get(usage, "output_token_details", "reasoning_tokens"),
        nested_get(usage, "output_token_details", "reasoning"),
    ]
    return sum(number(value) for value in candidates)


def estimate_visible_tokens(value: Any) -> float:
    """Very rough fallback for visible text only; not a substitute for provider usage."""
    if value is None:
        return 0.0
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False)
    return float(len(re.findall(r"\w+|[^\w\s]", text)))


def extract_answer(record: dict[str, Any]) -> Any:
    if "answer" in record:
        return record.get("answer")
    output = record.get("output")
    if isinstance(output, dict):
        return output.get("answer") or output.get("formatted") or output
    return output


def extract_top_level_usage(record: dict[str, Any]) -> dict[str, Any]:
    usage = record.get("token_usage")
    return usage if isinstance(usage, dict) else {}


def parse_jsonl(path: Path, method: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    call_rows = []
    experiment = path.parent.name

    with path.open() as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            record = json.loads(line)
            usage = extract_top_level_usage(record)
            calls = usage.get("calls") if isinstance(usage.get("calls"), list) else []
            answer = extract_answer(record)
            messages = record.get("messages")

            row = {
                "method": method,
                "experiment": experiment,
                "path": str(path),
                "line_no": line_no,
                "ground_truth_id": record.get("id") or record.get("ground_truth_id") or record.get("prediction_id"),
                "question": record.get("question") or record.get("original_question"),
                "time_taken": number(record.get("time_taken") or record.get("elapsed")),
                "has_provider_usage": bool(usage),
                "usage_source": usage.get("source") if usage else None,
                "usage_estimated": usage.get("estimated") if usage else None,
                "calls_count": len(calls),
                "prompt_tokens": number(usage.get("prompt_tokens")),
                "completion_tokens": number(usage.get("completion_tokens")),
                "total_tokens": number(usage.get("total_tokens")),
                "reasoning_tokens": extract_reasoning_tokens(usage),
                "visible_answer_token_estimate": estimate_visible_tokens(answer),
                "visible_message_token_estimate": estimate_visible_tokens(messages),
                "answer_preview": str(answer)[:300] if answer is not None else None,
            }
            if not row["total_tokens"] and (row["prompt_tokens"] or row["completion_tokens"]):
                row["total_tokens"] = row["prompt_tokens"] + row["completion_tokens"]
            rows.append(row)

            for call_idx, call in enumerate(calls):
                call_usage = call.get("usage") if isinstance(call, dict) else {}
                call_usage = call_usage if isinstance(call_usage, dict) else {}
                call_row = {
                    "method": method,
                    "experiment": experiment,
                    "ground_truth_id": row["ground_truth_id"],
                    "call_idx": call_idx,
                    "model": call.get("model") if isinstance(call, dict) else None,
                    "source": call.get("source") if isinstance(call, dict) else None,
                    "estimated": call.get("estimated") if isinstance(call, dict) else None,
                    "prompt_tokens": number(call_usage.get("prompt_tokens") or call_usage.get("input_tokens")),
                    "completion_tokens": number(call_usage.get("completion_tokens") or call_usage.get("output_tokens")),
                    "total_tokens": number(call_usage.get("total_tokens")),
                    "reasoning_tokens": extract_reasoning_tokens(call_usage),
                }
                if not call_row["total_tokens"] and (call_row["prompt_tokens"] or call_row["completion_tokens"]):
                    call_row["total_tokens"] = call_row["prompt_tokens"] + call_row["completion_tokens"]
                call_rows.append(call_row)

    return pd.DataFrame(rows), pd.DataFrame(call_rows)


frames = []
call_frames = []
for method, path in selected_raw_files[["method", "path"]].drop_duplicates().itertuples(index=False):
    df, calls = parse_jsonl(Path(path), method)
    frames.append(df)
    call_frames.append(calls)

raw_usage = pd.concat(frames, ignore_index=True)
raw_calls = pd.concat([df for df in call_frames if not df.empty], ignore_index=True) if any(not df.empty for df in call_frames) else pd.DataFrame()

raw_usage.head()

In [ ]:
coverage = (
    raw_usage.groupby(["method", "experiment"], as_index=False)
    .agg(
        examples=("ground_truth_id", "nunique"),
        rows=("ground_truth_id", "size"),
        rows_with_provider_usage=("has_provider_usage", "sum"),
        rows_with_nonzero_total=("total_tokens", lambda s: int((s > 0).sum())),
        total_provider_tokens=("total_tokens", "sum"),
        visible_answer_token_estimate=("visible_answer_token_estimate", "sum"),
        visible_message_token_estimate=("visible_message_token_estimate", "sum"),
    )
)
coverage["provider_usage_coverage"] = coverage["rows_with_provider_usage"] / coverage["rows"]
coverage

Rows where `provider_usage_coverage == 0` do not include reliable LLM token counters in the raw result file. The visible-token estimates are only a rough count of text that appears in the JSON and should not be treated as actual prompt/completion usage.

## 3. Summary by Method

In [ ]:
usage_summary = (
    raw_usage.groupby("method", as_index=False)
    .agg(
        examples=("ground_truth_id", "nunique"),
        rows=("ground_truth_id", "size"),
        rows_with_provider_usage=("has_provider_usage", "sum"),
        total_tokens_sum=("total_tokens", "sum"),
        prompt_tokens_sum=("prompt_tokens", "sum"),
        completion_tokens_sum=("completion_tokens", "sum"),
        reasoning_tokens_sum=("reasoning_tokens", "sum"),
        total_tokens_mean=("total_tokens", "mean"),
        total_tokens_median=("total_tokens", "median"),
        total_tokens_p95=("total_tokens", lambda s: s.quantile(0.95)),
        calls_mean=("calls_count", "mean"),
        time_taken_sum=("time_taken", "sum"),
        time_taken_mean=("time_taken", "mean"),
        visible_answer_token_estimate_sum=("visible_answer_token_estimate", "sum"),
        visible_message_token_estimate_sum=("visible_message_token_estimate", "sum"),
    )
)
usage_summary["provider_usage_coverage"] = usage_summary["rows_with_provider_usage"] / usage_summary["rows"]
usage_summary["tokens_per_second"] = usage_summary["total_tokens_sum"] / usage_summary["time_taken_sum"].replace(0, pd.NA)
usage_summary.sort_values("total_tokens_sum", ascending=False)

In [ ]:
plot_summary = usage_summary.copy().sort_values("total_tokens_sum", ascending=True)

fig, ax = plt.subplots(figsize=(9, 4.8))
ax.barh(plot_summary["method"], plot_summary["prompt_tokens_sum"], label="Prompt tokens", color="#4C78A8")
ax.barh(
    plot_summary["method"],
    plot_summary["completion_tokens_sum"],
    left=plot_summary["prompt_tokens_sum"],
    label="Completion tokens",
    color="#F58518",
)
ax.set_xlabel("Provider-reported tokens")
ax.set_title("Total Provider-Reported Token Usage by Method")
ax.grid(axis="x", alpha=0.25)
ax.legend()

for _, row in plot_summary.iterrows():
    ax.text(row["total_tokens_sum"] + max(plot_summary["total_tokens_sum"].max() * 0.01, 1), row["method"], f"{row['total_tokens_sum']:,.0f}", va="center")

plt.tight_layout()

In [ ]:
nonzero_usage = raw_usage[raw_usage["total_tokens"] > 0].copy()

fig, ax = plt.subplots(figsize=(9, 4.8))
if nonzero_usage.empty:
    ax.text(0.5, 0.5, "No provider token usage available", ha="center", va="center")
    ax.axis("off")
else:
    nonzero_usage.boxplot(column="total_tokens", by="method", ax=ax, grid=False)
    ax.set_yscale("log")
    ax.set_xlabel("Method")
    ax.set_ylabel("Provider-reported total tokens per question (log scale)")
    ax.set_title("Per-Question Token Distribution")
    plt.suptitle("")
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

## 4. Aligned Comparison by Ground-Truth ID

In [ ]:
token_pivot = raw_usage.pivot_table(
    index="ground_truth_id",
    columns="method",
    values="total_tokens",
    aggfunc="sum",
)

for baseline in ["fullcontext", "llmbased", "grasp"]:
    if "ours" in token_pivot.columns and baseline in token_pivot.columns:
        denom = token_pivot[baseline].replace(0, pd.NA)
        token_pivot[f"ours_over_{baseline}"] = token_pivot["ours"] / denom

token_pivot.head()

In [ ]:
if {"ours", "fullcontext"}.issubset(token_pivot.columns):
    comparable = token_pivot[(token_pivot["ours"] > 0) & (token_pivot["fullcontext"] > 0)].copy()
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(comparable["fullcontext"], comparable["ours"], alpha=0.75)
    lo = min(comparable[["fullcontext", "ours"]].min())
    hi = max(comparable[["fullcontext", "ours"]].max())
    ax.plot([lo, hi], [lo, hi], linestyle="--", color="#666666", linewidth=1)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Full context tokens")
    ax.set_ylabel("Ours tokens")
    ax.set_title("Per-Question Token Usage: Ours vs Full Context")
    ax.grid(alpha=0.25)
    plt.tight_layout()
else:
    print("Ours/fullcontext provider token usage is not available for aligned comparison.")

## 5. Token Usage by Question Type

In [ ]:
gt_qtypes = []
for task in TASKS:
    df = pd.read_csv(ANALYSIS_DIR / task / "results.csv")
    gt_qtypes.append(df[["ground_truth_id", "ground_truth_qtype"]].drop_duplicates())

gt_qtypes = pd.concat(gt_qtypes, ignore_index=True).drop_duplicates("ground_truth_id")
usage_with_qtype = raw_usage.merge(gt_qtypes, on="ground_truth_id", how="left")

qtype_summary = (
    usage_with_qtype.groupby(["ground_truth_qtype", "method"], as_index=False)
    .agg(
        examples=("ground_truth_id", "nunique"),
        total_tokens_sum=("total_tokens", "sum"),
        total_tokens_mean=("total_tokens", "mean"),
        prompt_tokens_mean=("prompt_tokens", "mean"),
        completion_tokens_mean=("completion_tokens", "mean"),
        provider_usage_rows=("has_provider_usage", "sum"),
    )
)
qtype_summary

In [ ]:
qtype_plot = qtype_summary.pivot(index="ground_truth_qtype", columns="method", values="total_tokens_mean")
ax = qtype_plot.plot(kind="bar", figsize=(10, 5), rot=25, width=0.78)
ax.set_ylabel("Mean provider-reported tokens per question")
ax.set_xlabel("Question type")
ax.set_title("Mean Token Usage by Question Type")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()

## 6. Call-Level Breakdown

In [ ]:
if raw_calls.empty:
    print("No per-call token usage was found in the selected raw files.")
else:
    call_summary = (
        raw_calls.groupby(["method", "model", "source"], dropna=False, as_index=False)
        .agg(
            calls=("call_idx", "size"),
            prompt_tokens_sum=("prompt_tokens", "sum"),
            completion_tokens_sum=("completion_tokens", "sum"),
            reasoning_tokens_sum=("reasoning_tokens", "sum"),
            total_tokens_sum=("total_tokens", "sum"),
            total_tokens_mean=("total_tokens", "mean"),
            total_tokens_p95=("total_tokens", lambda s: s.quantile(0.95)),
        )
        .sort_values("total_tokens_sum", ascending=False)
    )
    display(call_summary)

In [ ]:
if not raw_calls.empty:
    calls_by_question = (
        raw_calls.groupby(["method", "ground_truth_id"], as_index=False)
        .agg(calls=("call_idx", "size"), total_tokens=("total_tokens", "sum"))
        .sort_values("total_tokens", ascending=False)
    )
    display(calls_by_question.head(20))

## 7. Largest Token-Usage Questions

In [ ]:
largest_questions = (
    usage_with_qtype[usage_with_qtype["total_tokens"] > 0]
    .sort_values("total_tokens", ascending=False)
    [[
        "method", "ground_truth_id", "ground_truth_qtype", "total_tokens", "prompt_tokens",
        "completion_tokens", "reasoning_tokens", "calls_count", "time_taken", "question", "answer_preview",
    ]]
    .head(25)
)
largest_questions